In [ ]:
CATALOG = "spotify_etl"
SCHEMA = "bronze"
TABLE = ""  # set per notebook

dbutils.widgets.text("raw_base_path", "/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw", "RAW base path")
RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")

In [ ]:
import os, json, re

def collect():
    rows = []
    entity_path = f"{RAW_BASE_PATH}/playlist_tracks"
    if not os.path.exists(entity_path): return rows
    meta_map = {}
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.endswith("_meta.json"):
                m = re.match(r"page_(\d+)_meta\.json", fn)
                if m:
                    with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                        meta = json.load(f)
                    pid = meta.get("playlist_id")
                    if pid: meta_map[int(m.group(1))] = pid
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.startswith("page_") and fn.endswith(".json") and not fn.endswith("_meta.json"):
                m = re.match(r"page_(\d+)\.json", fn)
                page_idx = int(m.group(1)) if m else None
                with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                    data = json.load(f)
                page_pid = meta_map.get(page_idx)
                if page_pid is None:
                    href = data.get("href", "")
                    m2 = re.search(r"playlists/([^/]+)/tracks", href)
                    page_pid = m2.group(1) if m2 else None
                if page_pid is None: continue
                for item in data.get("items", []):
                    track = item.get("track")
                    if not track or not track.get("id"): continue
                    artist_ids = [a.get("id") for a in (track.get("artists") or []) if a.get("id")]
                    rows.append({
                        "playlist_id": page_pid, "track_id": track.get("id"),
                        "track_name": track.get("name"), "artist_ids": artist_ids,
                        "album_id": (track.get("album") or {}).get("id"),
                        "added_at": item.get("added_at"),
                        "added_by": (item.get("added_by") or {}).get("id"),
                        "duration_ms": track.get("duration_ms"), "popularity": track.get("popularity"),
                    })
    return rows

rows = collect()
if rows:
    df = spark.createDataFrame(rows)
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")
else:
    print(f"No data for {TABLE}")
